In [ ]:
import os, re, pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import pandas as pd

In [ ]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU detected:", gpus)
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Running on CPU.")

In [ ]:
DATASET_PATH = "../data/raw/fake_job_postings.csv"

DATA_DIR = "../data/processed"

MODEL_DIR = "../models"

JOINT_MODEL_PATH = os.path.join(MODEL_DIR, "joint_model_final.keras")
EXTRACTOR_PATH = os.path.join(MODEL_DIR, "bilstm_model_final.keras")
MAXOUT_PATH = os.path.join(MODEL_DIR, "maxout_model_final.keras")

TOKENIZER_PATH = os.path.join(DATA_DIR, "tokenizer.pkl")

In [ ]:
# ── Text Cleaning ──────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """Remove HTML, URLs, emails, special chars from real job posting text."""
    if not isinstance(text, str):
        text = str(text) if text is not None else ""
    text = text.lower()
    text = re.sub(r"<[^>]+>",               " ", text)   # HTML tags
    text = re.sub(r"&[a-z]+;",              " ", text)   # HTML entities
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"\S+@\S+",               " ", text)   # emails
    text = re.sub(r"[^a-z0-9\s]",           " ", text)   # special chars
    text = re.sub(r"\s+",                   " ", text).strip()
    return text



In [ ]:
# ── Custom Maxout Layer ────────────────────────────────────────────────────────
class MaxoutLayer(layers.Layer):
    """
    Maxout activation (Goodfellow et al., 2013).
    Each output unit takes the max over `num_pieces` linear projections.
    Piecewise-linear activation — expressive and works well with Dropout.
    """
    def __init__(self, units: int, num_pieces: int = 2, l2: float = 1e-4, **kwargs):
        super().__init__(**kwargs)
        self.units      = units
        self.num_pieces = num_pieces
        self.l2_val     = l2

    def build(self, input_shape):
        input_dim = int(input_shape[-1])
        reg = regularizers.l2(self.l2_val)
        self.W = self.add_weight(
            name="W", shape=(input_dim, self.units * self.num_pieces),
            initializer="glorot_uniform", regularizer=reg, trainable=True)
        self.b = self.add_weight(
            name="b", shape=(self.units * self.num_pieces,),
            initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        z = tf.matmul(inputs, self.W) + self.b
        z = tf.reshape(z, (-1, self.units, self.num_pieces))
        return tf.reduce_max(z, axis=-1)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.units, "num_pieces": self.num_pieces, "l2": self.l2_val})
        return cfg

In [ ]:

# ── Build Joint Model ──────────────────────────────────────────────────────────
def build_joint_model(vocab_size, embedding_dim, lstm_units, max_seq_len, l2=1e-4):
    """
    Joint BiLSTM + Maxout model trained end-to-end.

    Architecture
    ────────────
    Input (512,)
      -> Embedding(20000, 128)
      -> SpatialDropout1D(0.2)           drops entire word vectors
      -> Bidirectional(LSTM(128))        -> 256-dim semantic embedding
      -> Dropout(0.4) -> BatchNorm
      -> MaxoutLayer(256) -> Dropout(0.4)
      -> MaxoutLayer(128) -> Dropout(0.3)
      -> MaxoutLayer(64)  -> Dropout(0.2)
      -> Dense(1, sigmoid)
    """
    reg = regularizers.l2(l2)
    inp = layers.Input(shape=(max_seq_len,), name="token_input")

    x = layers.Embedding(vocab_size, embedding_dim, name="embedding")(inp)
    x = layers.SpatialDropout1D(0.2, name="spatial_dropout")(x)

    x = layers.Bidirectional(
            layers.LSTM(lstm_units,
                        kernel_regularizer=reg, recurrent_regularizer=reg,
                        name="lstm"),
            name="bilstm_layer"
        )(x)

    x = layers.Dropout(0.4, name="dropout_1")(x)
    x = layers.BatchNormalization(name="bn_1")(x)

    x = MaxoutLayer(256, num_pieces=2, l2=l2, name="maxout_1")(x)
    x = layers.Dropout(0.4, name="dropout_2")(x)

    x = MaxoutLayer(128, num_pieces=2, l2=l2, name="maxout_2")(x)
    x = layers.Dropout(0.3, name="dropout_3")(x)

    x = MaxoutLayer(64,  num_pieces=2, l2=l2, name="maxout_3")(x)
    x = layers.Dropout(0.2, name="dropout_4")(x)

    out = layers.Dense(1, activation="sigmoid", kernel_regularizer=reg, name="output")(x)

    model = models.Model(inputs=inp, outputs=out, name="Joint_BiLSTM_Maxout")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy",
                 tf.keras.metrics.AUC(name="auc"),
                 tf.keras.metrics.Precision(name="precision"),
                 tf.keras.metrics.Recall(name="recall")]
    )
    return model



In [ ]:

# ── Data Loading & Preprocessing ──────────────────────────────────────────────
def load_and_preprocess():
    os.makedirs(DATA_DIR,  exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    required = ["X_train.npy","X_val.npy","X_test.npy",
                "y_train.npy","y_val.npy","y_test.npy","tokenizer.pkl"]
    arrays_ok = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required)

    if arrays_ok:
        X_check = np.load(os.path.join(DATA_DIR, "X_train.npy"))
        if X_check.shape[1] != MAX_SEQUENCE_LENGTH:
            print(f"[DATA] Seq length mismatch ({X_check.shape[1]} vs "
                  f"{MAX_SEQUENCE_LENGTH}). Re-running preprocessing.")
            arrays_ok = False

    if arrays_ok:
        print("[DATA] Loading existing preprocessed arrays ...")
        X_train = np.load(os.path.join(DATA_DIR, "X_train.npy"))
        X_val   = np.load(os.path.join(DATA_DIR, "X_val.npy"))
        X_test  = np.load(os.path.join(DATA_DIR, "X_test.npy"))
        y_train = np.load(os.path.join(DATA_DIR, "y_train.npy"))
        y_val   = np.load(os.path.join(DATA_DIR, "y_val.npy"))
        y_test  = np.load(os.path.join(DATA_DIR, "y_test.npy"))
        with open(TOKENIZER_PATH, "rb") as f:
            tokenizer = pickle.load(f)
        print(f"      Train:{X_train.shape}  Val:{X_val.shape}  Test:{X_test.shape}")
        return X_train, X_val, X_test, y_train, y_val, y_test, tokenizer

    print("[DATA] Running full preprocessing on EMSCAD dataset ...")
    df = pd.read_csv(DATASET_PATH)
    print(f"      Dataset shape : {df.shape}")

    for col in TEXT_COLS:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].fillna("")

    labels = df["fraudulent"].astype(int).values
    print(f"      Real (0) : {(labels==0).sum()}  |  Fake (1) : {(labels==1).sum()}")
    print(f"      Imbalance: {(labels==0).sum()/(labels==1).sum():.1f}:1")

    df["merged_text"] = (df["title"]           + " " +
                         df["company_profile"] + " " +
                         df["description"]     + " " +
                         df["requirements"]    + " " +
                         df["benefits"])

    print("      Cleaning text (HTML, URLs, emails, special chars) ...")
    df["cleaned_text"] = df["merged_text"].apply(clean_text)

    wc = df["cleaned_text"].apply(lambda x: len(x.split()))
    print(f"      Word count    : mean={wc.mean():.0f}  "
          f"median={wc.median():.0f}  p90={wc.quantile(0.9):.0f}  max={wc.max()}")

    print(f"      Tokenizing (vocab={VOCAB_SIZE}) ...")
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
    tokenizer.fit_on_texts(df["cleaned_text"].tolist())

    seqs   = tokenizer.texts_to_sequences(df["cleaned_text"].tolist())
    padded = pad_sequences(seqs, maxlen=MAX_SEQUENCE_LENGTH,
                           padding="post", truncating="post")
    print(f"      Padded shape  : {padded.shape}")

    X_tv, X_test, y_tv, y_test = train_test_split(
        padded, labels, test_size=0.15, random_state=RANDOM_STATE, stratify=labels)
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=0.15/0.85, random_state=RANDOM_STATE, stratify=y_tv)

    print(f"      Train:{X_train.shape}  Val:{X_val.shape}  Test:{X_test.shape}")
    print(f"      Train fakes: {y_train.sum()} / {len(y_train)}  "
          f"| Val fakes: {y_val.sum()} / {len(y_val)}  "
          f"| Test fakes: {y_test.sum()} / {len(y_test)}")

    for name, arr in [("X_train",X_train),("X_val",X_val),("X_test",X_test),
                      ("y_train",y_train),("y_val",y_val),("y_test",y_test)]:
        np.save(os.path.join(DATA_DIR, f"{name}.npy"), arr)
    with open(TOKENIZER_PATH, "wb") as f:
        pickle.dump(tokenizer, f)
    print("      All arrays and tokenizer saved.")

    return X_train, X_val, X_test, y_train, y_val, y_test, tokenizer



In [ ]:

# ── Training ───────────────────────────────────────────────────────────────────
def train(X_train, X_val, y_train, y_val):
    print("\n[TRAIN] Building joint BiLSTM + Maxout model ...")
    model = build_joint_model(VOCAB_SIZE, EMBEDDING_DIM, LSTM_UNITS,
                              MAX_SEQUENCE_LENGTH, L2)
    model.summary()

    # Class weights — critical for 19.6:1 imbalance
    cw_arr       = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
    class_weight = {0: cw_arr[0], 1: cw_arr[1]}
    print(f"\n      Class weights: Real={class_weight[0]:.3f}  Fake={class_weight[1]:.3f}")
    print(f"\n[TRAIN] Training up to {EPOCHS} epochs (patience={PATIENCE}) ...\n")

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE,
                      restore_best_weights=True, mode="min", verbose=1),
        ModelCheckpoint(JOINT_MODEL_PATH, monitor="val_loss",
                        save_best_only=True, mode="min", verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                          min_lr=1e-6, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        validation_data = (X_val, y_val),
        epochs          = EPOCHS,
        batch_size      = BATCH_SIZE,
        class_weight    = class_weight,
        callbacks       = callbacks
    )
    return model, history


In [ ]:

# ── Evaluate ───────────────────────────────────────────────────────────────────
def evaluate(model, X_test, y_test, history):
    print("\n[EVAL] Test set evaluation ...")
    results      = model.evaluate(X_test, y_test, verbose=0)
    metric_names = [m.name for m in model.metrics]
    for name, val in zip(metric_names, results):
        print(f"      {name:15s}: {val:.4f}")

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob > THRESHOLD).astype(int)

    auc_score = roc_auc_score(y_test, y_prob)
    print(f"      {'roc_auc':15s}: {auc_score:.4f}")

    pred_dist = {("Real" if u==0 else "Fake"): int(c)
                 for u,c in zip(*np.unique(y_pred, return_counts=True))}
    print(f"\n      Prediction spread : {pred_dist}")
    if len(pred_dist) == 1:
        print("  [WARNING] Model predicts only one class!")

    print(f"\n  Classification Report (threshold={THRESHOLD}):")
    print(classification_report(y_test, y_pred,
                                target_names=["Real Job (0)", "Fake Job (1)"]))

    cm = confusion_matrix(y_test, y_pred)
    print("  Confusion Matrix:")
    print(f"                    Predicted Real   Predicted Fake")
    print(f"  Actual Real  :         {cm[0,0]:5d}            {cm[0,1]:5d}")
    print(f"  Actual Fake  :         {cm[1,0]:5d}            {cm[1,1]:5d}")

    # Training curves
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (tr, va), title in zip(
        axes,
        [("accuracy","val_accuracy"),("loss","val_loss"),("auc","val_auc")],
        ["Accuracy","Loss","AUC"]
    ):
        ax.plot(history.history[tr], label="Train", linewidth=2)
        ax.plot(history.history[va], label="Val",   linewidth=2)
        ax.set_title(f"BiLSTM+Maxout — {title}", fontsize=12)
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.suptitle("Training History — EMSCAD Real Dataset", fontsize=13, y=1.02)
    plt.tight_layout()
    plot_path = os.path.join(MODEL_DIR, "training_history.png")
    plt.savefig(plot_path, bbox_inches="tight", dpi=150)
    plt.close()
    print(f"\n  Training plot saved -> {plot_path}")


In [ ]:

# ── Save Sub-Models ────────────────────────────────────────────────────────────
def save_sub_models(joint_model):
    print("\n[SAVE] Extracting sub-models ...")

    bilstm_out = joint_model.get_layer("bilstm_layer").output
    extractor  = Model(inputs=joint_model.input, outputs=bilstm_out,
                       name="BiLSTM_Extractor")
    extractor.save(EXTRACTOR_PATH)
    print(f"      BiLSTM extractor  -> {EXTRACTOR_PATH}  (output: {extractor.output_shape})")

    bilstm_dim = extractor.output_shape[-1]  # 256
    feat_input = layers.Input(shape=(bilstm_dim,), name="features_input")
    x   = joint_model.get_layer("maxout_1")(feat_input)
    x   = joint_model.get_layer("dropout_2")(x)
    x   = joint_model.get_layer("maxout_2")(x)
    x   = joint_model.get_layer("dropout_3")(x)
    x   = joint_model.get_layer("maxout_3")(x)
    x   = joint_model.get_layer("dropout_4")(x)
    out = joint_model.get_layer("output")(x)
    maxout_model = Model(inputs=feat_input, outputs=out, name="Maxout_Classifier")
    maxout_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    maxout_model.save(MAXOUT_PATH)
    print(f"      Maxout classifier -> {MAXOUT_PATH}")


In [ ]:
if __name__ == "__main__":
    main()
